In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_7.py ──
"""
Shared infrastructure for Exercise 7 — Transfer Learning.

Contains: CIFAR-10 data loading, feature visualisation helpers,
ExperimentTracker/ModelRegistry setup, training harness.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex7_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — CIFAR-10 (full 50K, resized for ResNet-18)
# ════════════════════════════════════════════════════════════════════════
# ResNet-18 was designed for 224x224 ImageNet images. CIFAR-10 is 32x32.
# We resize to 96x96: ResNet's strided convolutions shrink spatial dims
# by 32x, so 32x32 would collapse to 1x1 before final pooling. 96x96
# gives a 3x3 final feature map — enough spatial information to learn.

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Standard transforms: ImageNet normalisation + resize for ResNet
train_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomCrop(INPUT_SIZE, padding=8),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_cifar10() -> tuple[
    torchvision.datasets.CIFAR10,
    torchvision.datasets.CIFAR10,
    DataLoader,
    DataLoader,
]:
    """Load CIFAR-10 with ImageNet-style transforms.

    Returns:
        train_set, val_set, train_loader, val_loader
    """
    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

    print(
        f"CIFAR-10 (full): train={len(train_set)}, val={len(val_set)}, "
        f"classes={N_CLASSES}"
    )
    print(f"  Input size: {INPUT_SIZE}x{INPUT_SIZE} (resized for ResNet-18)")
    print(f"  Classes: {CLASS_NAMES}")

    return train_set, val_set, train_loader, val_loader


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transfer.db"
    registry_db = "sqlite:///mlfp05_transfer_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transfer_learning", registry, True


def init_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS — shared by all technique files
# ════════════════════════════════════════════════════════════════════════


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Train a model and log everything to ExperimentTracker.

    Returns:
        train_losses, val_accs, train_accs (per-epoch)
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n-- {name} --  trainable params: {n_trainable:,} / {n_total:,}")

    opt = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_losses: list[float] = []
    val_accs: list[float] = []
    train_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "trainable_params": str(n_trainable),
                "total_params": str(n_total),
                "epochs": str(epochs),
                "lr": str(lr),
                "batch_size": str(tr_loader.batch_size),
                "dataset_size": str(len(tr_loader.dataset)),
            }
        )

        for epoch in range(epochs):
            # -- Training --
            model.train()
            batch_losses = []
            correct = total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
                correct += int((logits.argmax(dim=-1) == yb).sum().item())
                total += int(yb.size(0))
            train_losses.append(float(np.mean(batch_losses)))
            train_accs.append(correct / total)
            scheduler.step()

            # -- Validation --
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
            val_accs.append(correct / total)

            await run.log_metrics(
                {
                    "train_loss": train_losses[-1],
                    "train_acc": train_accs[-1],
                    "val_acc": val_accs[-1],
                },
                step=epoch + 1,
            )

            print(
                f"  epoch {epoch + 1}/{epochs}  "
                f"loss={train_losses[-1]:.4f}  "
                f"train_acc={train_accs[-1]:.3f}  "
                f"val_acc={val_accs[-1]:.3f}"
            )

        await run.log_metrics(
            {
                "final_val_acc": val_accs[-1],
                "best_val_acc": max(val_accs),
                "final_train_loss": train_losses[-1],
            }
        )

    return train_losses, val_accs, train_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            tr_loader,
            vl_loader,
            epochs,
            lr,
        )
    )


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Register a trained model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="val_acc", value=val_acc),
            MetricSpec(name="final_loss", value=final_loss),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={val_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, val_acc, final_loss))


# ════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION & VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════


def extract_features(
    model: nn.Module,
    loader: DataLoader,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features from the penultimate layer (before fc head).

    Works for both ResNet (avgpool hook) and Sequential models.
    """
    model.eval()
    hook_features: list[torch.Tensor] = []
    labels: list[np.ndarray] = []

    def hook_fn(module, inp, out):
        del module, inp  # PyTorch hook protocol args; not used here
        hook_features.append(out.flatten(1).detach().cpu())

    # ResNet: hook into avgpool; Sequential: second-to-last layer
    if hasattr(model, "avgpool"):
        avgpool = model.avgpool
        assert isinstance(
            avgpool, nn.Module
        ), f"expected nn.Module for avgpool, got {type(avgpool).__name__}"
        handle = avgpool.register_forward_hook(hook_fn)
    else:
        assert isinstance(
            model, nn.Sequential
        ), f"hook fallback requires nn.Sequential, got {type(model).__name__}"
        handle = model[-3].register_forward_hook(hook_fn)

    with torch.no_grad():
        collected = 0
        for xb, yb in loader:
            if collected >= max_samples:
                break
            xb = xb.to(device)
            model(xb)
            labels.append(yb.numpy())
            collected += len(yb)

    handle.remove()
    features_np = torch.cat(hook_features, dim=0).numpy()[:max_samples]
    labels_np = np.concatenate(labels)[:max_samples]
    return features_np, labels_np


def compute_tsne(features: np.ndarray, perplexity: int = 30) -> np.ndarray:
    """Run t-SNE dimensionality reduction to 2D."""
    # sklearn 1.5+ renamed n_iter → max_iter in TSNE.__init__
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=500, random_state=42)
    return tsne.fit_transform(features)


def cluster_quality(coords: np.ndarray, labels: np.ndarray) -> float:
    """Cluster quality: ratio of intra-class to inter-class distance (lower = better)."""
    intra = []
    centroids = []
    for c in range(N_CLASSES):
        mask = labels == c
        if mask.sum() < 2:
            continue
        pts = coords[mask]
        centroid = pts.mean(axis=0)
        centroids.append(centroid)
        intra.append(np.mean(np.linalg.norm(pts - centroid, axis=1)))
    centroids_arr = np.array(centroids)
    inter = np.mean(
        [
            np.linalg.norm(centroids_arr[i] - centroids_arr[j])
            for i in range(len(centroids_arr))
            for j in range(i + 1, len(centroids_arr))
        ]
    )
    avg_intra = np.mean(intra)
    return float(avg_intra / inter) if inter > 0 else float("inf")


def plot_tsne(
    coords: np.ndarray,
    labels: np.ndarray,
    title: str,
    output_path: str | Path,
) -> None:
    """Create and save a t-SNE scatter plot coloured by class."""
    fig = go.Figure()
    for c in range(N_CLASSES):
        mask = labels == c
        fig.add_trace(
            go.Scatter(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=CLASS_NAMES[c],
                marker=dict(size=4, opacity=0.6),
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance."""
    return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(metrics=metrics, x_label="Epoch", y_label="Value")
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters in a model."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 7, Part 3: Data Efficiency Experiment
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Run a controlled data efficiency experiment: train a transfer
#     model on 10%, 25%, 50%, and 100% of training data
#   - Quantify how transfer learning bends the labelling cost curve
#   - Plot data efficiency curves comparing transfer vs from-scratch
#   - Answer the business question: "How many images do we need to label?"
#   - Calculate cost savings in a real Singapore business scenario
#
# PREREQUISITES: Part 1 (baseline), Part 2 (transfer learning).
# ESTIMATED TIME: ~25 min
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision



## THEORY — The Labelling Bottleneck


In [ ]:
# In production ML, the biggest cost isn't compute — it's labelled data.
#
# Consider what labelling costs in practice:
#   - Simple image classification: S$0.50-1.00 per image
#   - Medical image annotation: S$5-20 per image (specialist required)
#   - Autonomous driving frames: S$50-200 per frame (3D bounding boxes)
#
# If you need 50,000 labelled images at S$1 each, that's S$50,000 just
# for data — before you've written a single line of code.
#
# Transfer learning bends this cost curve. By reusing features from a
# pre-trained model, you can achieve 80-90% of the full-data accuracy
# with only 10-25% of the labelled data. This experiment quantifies
# exactly where that sweet spot is.
#
# The key question for any ML project: "What's the minimum amount of
# labelled data that gives us acceptable accuracy?" This experiment
# gives you the data to answer it.



In [ ]:
print("\n" + "=" * 70)
print("  PART 3: Data Efficiency Experiment")
print("=" * 70)



## TASK 1 — Load data and set up engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()



## TASK 2 — Define model builders


Frozen ResNet-18 backbone + fresh classification head.


From-scratch CNN baseline.


In [ ]:
def build_transfer_resnet(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Load pre-trained ResNet-18, freeze backbone, replace fc head
    # Steps:
    #   1. Load weights: torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    #   2. Create model with those weights (try/except for offline fallback)
    #   3. Freeze all params: for p in model.parameters(): p.requires_grad = False
    #   4. Replace model.fc with nn.Linear(model.fc.in_features, n_classes)
    # Hint: Same pattern as Part 2's build_transfer_resnet
    ____


def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Build a 3-layer CNN identical to Part 1
    # Hint: nn.Sequential with Conv2d->BN->ReLU->Pool blocks
    #       then AdaptiveAvgPool2d(1)->Flatten->Dropout(0.3)->Linear(128, n_classes)
    ____



## TASK 3 — Run the data efficiency experiment


Train one model on a fraction of data, return (accuracy, n_samples).


In [ ]:
# For each data fraction (10%, 25%, 50%, 100%), we train BOTH a transfer
# model and a from-scratch model, then compare. This gives us two
# curves that show exactly where transfer learning helps most.

DATA_FRACTIONS = [0.10, 0.25, 0.50, 1.0]
EFF_EPOCHS = 4  # Shorter training for sub-experiments

transfer_results: dict[float, float] = {}
scratch_results: dict[float, float] = {}

rng = np.random.default_rng(42)


async def _run_efficiency_trial(
    frac: float,
    model_builder,
    model_name: str,
) -> tuple[float, int]:
    # TODO: Implement the efficiency trial
    # Steps:
    #   1. Compute n_samples = int(len(train_set) * frac)
    #   2. Sample random indices: rng.choice(len(train_set), size=n_samples, replace=False).tolist()
    #   3. Create Subset(train_set, indices) and DataLoader
    #   4. Build model, move to device, get trainable params
    #   5. Create Adam optimizer with lr=1e-3
    #   6. Log to ExperimentTracker with run_name=f"{model_name}_{int(frac*100)}pct"
    #   7. Train for EFF_EPOCHS: forward -> cross_entropy -> backward -> step
    #   8. Evaluate on full val_loader: count correct/total
    #   9. Return (accuracy, n_samples)
    # Hint: async with tracker.track(experiment=exp_name, run_name=run_name) as run:
    # Hint: await run.log_params({...}), await run.log_metric("val_acc", acc)
    n_samples = int(len(train_set) * frac)
    ____

    return ____, n_samples


print("\n" + "=" * 70)
print("  DATA EFFICIENCY EXPERIMENT")
print("=" * 70)

for frac in DATA_FRACTIONS:
    # Transfer model
    t_acc, n_samples = asyncio.run(
        _run_efficiency_trial(frac, build_transfer_resnet, "transfer")
    )
    transfer_results[frac] = t_acc

    # From-scratch model
    s_acc, _ = asyncio.run(_run_efficiency_trial(frac, build_scratch_cnn, "scratch"))
    scratch_results[frac] = s_acc

    print(
        f"  {frac * 100:5.0f}% data ({n_samples:>5,} samples): "
        f"transfer={t_acc:.4f}  scratch={s_acc:.4f}  "
        f"gap={t_acc - s_acc:+.4f}"
    )



### Checkpoint 1


In [ ]:
assert len(transfer_results) == len(
    DATA_FRACTIONS
), "Should have transfer results for all fractions"
assert len(scratch_results) == len(
    DATA_FRACTIONS
), "Should have scratch results for all fractions"
assert (
    transfer_results[0.10] > 0.15
), f"Transfer with 10% data should beat random (acc={transfer_results[0.10]:.3f})"
# INTERPRETATION: Transfer learning shows diminishing returns as data
# increases — the gap between 10% and 100% is smaller than the scratch
# model's gap. Pre-trained features already capture general visual
# patterns, so additional data helps but isn't as critical.
print("\n--- Checkpoint 1 passed --- efficiency experiment complete\n")



## TASK 4 — Visualise: Data efficiency curves


In [ ]:
# The money chart: accuracy vs percentage of training data for both
# approaches. This is the chart you show to the VP of Engineering.

fracs = sorted(transfer_results.keys())
transfer_accs_by_frac = [transfer_results[f] for f in fracs]
scratch_accs_by_frac = [scratch_results[f] for f in fracs]
pct_labels = [f * 100 for f in fracs]

# TODO: Create a Plotly figure with two traces:
#   1. Transfer learning curve (lines+markers, color="#2196F3")
#   2. From-scratch curve (lines+markers, dash="dash", color="#FF5722")
# Add annotations for the 10% data points on both curves
# Hint: fig = go.Figure()
# Hint: fig.add_trace(go.Scatter(x=pct_labels, y=transfer_accs_by_frac, ...))
# Hint: fig.add_annotation(x=10, y=transfer_results[0.10], text=..., showarrow=True)
fig = go.Figure()
____

fig.update_layout(
    title="Data Efficiency: Transfer Learning vs From-Scratch",
    xaxis_title="% of CIFAR-10 Training Data",
    yaxis_title="Validation Accuracy",
    template="plotly_white",
    legend=dict(x=0.6, y=0.15),
    width=800,
    height=500,
)

eff_path = OUTPUT_DIR / "03_data_efficiency.html"
fig.write_html(str(eff_path))
print(f"  Saved: {eff_path}")



### Checkpoint 2


In [ ]:
assert (eff_path).exists(), "Data efficiency plot should be saved"
print("--- Checkpoint 2 passed --- efficiency curves plotted\n")



## TASK 5 — Visualise: Accuracy gap and diminishing returns


In [ ]:
# The gap between transfer and scratch narrows as data increases.
# This shows that transfer learning's biggest value is with LIMITED data.

# TODO: Compute gaps and create a bar chart
# Steps:
#   1. gaps = [t - s for t, s in zip(transfer_accs_by_frac, scratch_accs_by_frac)]
#   2. Create go.Figure() with go.Bar trace
#   3. x = [f"{p:.0f}%" for p in pct_labels], y = [g * 100 for g in gaps]
#   4. Color bars green if gap > 0, red otherwise
#   5. Save to OUTPUT_DIR / "03_accuracy_gap.html"
# Hint: marker_color=["#4CAF50" if g > 0 else "#F44336" for g in gaps]
gaps = [t - s for t, s in zip(transfer_accs_by_frac, scratch_accs_by_frac)]
____

gap_path = OUTPUT_DIR / "03_accuracy_gap.html"
fig_gap.write_html(str(gap_path))
print(f"  Saved: {gap_path}")



## TASK 6 — Apply: The VP of Engineering at Grab Asks "How Many Images?"


In [ ]:
# SCENARIO: You're the ML lead at Grab Singapore. The VP of Engineering
# asks: "We want to build an image classifier for food delivery photos.
# How many images do we need to label? What will it cost?"
#
# You use this data efficiency experiment to answer concretely.

print("\n" + "=" * 70)
print("  APPLY: Grab Singapore — 'How many images do we need to label?'")
print("=" * 70)

COST_PER_LABEL = 0.80  # S$ per image label (food photo classification)
TOTAL_AVAILABLE = 50000  # Total unlabelled images available

# TODO: Print the cost-accuracy trade-off table
# Steps:
#   1. Loop through fracs, compute n_images and label_cost for each
#   2. Print a formatted table with columns: Data%, Images, Transfer acc,
#      Scratch acc, Label Cost, Transfer Saves
# Hint: n_images = int(TOTAL_AVAILABLE * frac)
# Hint: label_cost = n_images * COST_PER_LABEL
print(f"\n  === Cost-Accuracy Trade-off Analysis ===")
print(f"  Labelling cost: S${COST_PER_LABEL:.2f} per image")
print(f"  Unlabelled pool: {TOTAL_AVAILABLE:,} food delivery photos")
____

# TODO: Find the sweet spot where transfer reaches 90% of max accuracy
# Steps:
#   1. max_transfer_acc = transfer_results[1.0]
#   2. sweet_spot_threshold = 0.90 * max_transfer_acc
#   3. Loop through fracs to find first frac where accuracy >= threshold
#   4. Calculate savings vs labelling the full dataset
# Hint: sweet_spot_frac = None; iterate and break when found
max_transfer_acc = transfer_results[1.0]
sweet_spot_threshold = 0.90 * max_transfer_acc
sweet_spot_frac = None
____

if sweet_spot_frac is not None:
    sweet_n = int(TOTAL_AVAILABLE * sweet_spot_frac)
    sweet_cost = sweet_n * COST_PER_LABEL
    full_cost = TOTAL_AVAILABLE * COST_PER_LABEL
    savings = full_cost - sweet_cost
    print(f"\n  SWEET SPOT: {sweet_spot_frac * 100:.0f}% of data ({sweet_n:,} images)")
    print(
        f"  Reaches {transfer_results[sweet_spot_frac]:.1%} accuracy "
        f"(90% of maximum {max_transfer_acc:.1%})"
    )
    print(f"  Label cost: S${sweet_cost:,.0f} vs S${full_cost:,.0f} for full dataset")
    print(f"  SAVINGS: S${savings:,.0f}")
else:
    print(f"\n  All fractions tested achieve >=90% of maximum accuracy.")

print()
print(f"  RECOMMENDATION TO VP:")
print(
    f"  'Start with {int(TOTAL_AVAILABLE * 0.25):,} labelled images "
    f"(S${int(TOTAL_AVAILABLE * 0.25 * COST_PER_LABEL):,})."
)
print(f"   Use transfer learning with ResNet-18. If accuracy is insufficient,")
print(f"   label more images in batches of 5,000 until you reach the target.")
print(f"   Transfer learning means we never need to label all 50,000 images.'")



### Checkpoint 3


In [ ]:
assert (
    sweet_spot_frac is not None or len(fracs) > 0
), "Should identify a sweet spot or have results"
# INTERPRETATION: The data efficiency curve directly answers the VP's
# question with concrete numbers: how many images to label, how much
# it costs, and where the diminishing returns kick in. This is how ML
# engineers translate technical results into business decisions.
print("\n--- Checkpoint 3 passed --- business analysis complete\n")



## CLEANUP


In [ ]:
await conn.close()



## REFLECTION


[x] Ran data efficiency experiment across 4 data fractions
  [x] Transfer with 10% data: {transfer_results[0.10]:.1%} accuracy
  [x] Transfer with 100% data: {transfer_results[1.0]:.1%} accuracy
  [x] Scratch with 10% data: {scratch_results[0.10]:.1%} accuracy
  [x] Plotted data efficiency curves (transfer vs scratch)
  [x] Identified the sweet spot: {sweet_spot_frac * 100:.0f}% of data for 90% of max accuracy
  [x] Calculated labelling cost savings for Grab Singapore scenario

  KEY INSIGHT: Transfer learning's biggest value is with LIMITED data.
  The gap between transfer and scratch is largest at 10-25% data, then
  narrows as data increases. This means:
    - With abundant data: transfer helps but isn't critical
    - With scarce data: transfer is transformative

  THE LABELLING BOTTLENECK EQUATION:
    Cost = (images needed) x (cost per label)
    Transfer learning reduces the first term by 4-10x.
    This is often the difference between a viable project and a shelved one.

  NEXT: Part 4 introduces adapter modules — a parameter-efficient
  alternative to full fine-tuning that bridges to M6's LoRA technique.


In [ ]:
print("\n" + "=" * 70)
print("  PART 3 COMPLETE — What You've Learned")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Training at 10%, 25%, 50%, 100% of data
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (Data efficiency — how small can we go?) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        models_by_frac[1.0],
        train_loader,
        _diag_loss,
        title="Data efficiency — how small can we go?",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [Cross-run comparison — all 4 data fractions]
# 100% data: RMS healthy, 87% val accuracy
#  50% data: RMS healthy, 84% val accuracy
#  25% data: train-val gap widening (overfit)
#  10% data: [CRITICAL] 52% val accuracy — too little data to generalise



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [STETHOSCOPE] The data-efficiency curve shows transfer
#     learning's power: 50% of data still gives ~97% of full
#     performance. Below 25%, diminishing returns kick in.
#     >> Decision rule: if you have >1000 labelled examples,
#        transfer learning + fine-tune works. If <500, try
#        adapter modules (ex_7/04) or few-shot methods.
#
#  [SCALING LAWS] This is the practical flipside of slide 5M
#     (scaling laws) — for downstream tasks with small data,
#     pretrained features + small fine-tune data = best ROI.

